# 02 — Huấn luyện YOLO11-seg từ `data/processing/` & Xuất Kết quả ra `data/output/` + Root

**Luồng thực thi chuẩn của Notebook này:**
1. **Đầu vào:** Đọc bộ dữ liệu ảnh đã tiền xử lý từ **`data/processing/`** (`train/` và `valid/`, được xuất bởi `01_pre_processing.ipynb`, khai báo trong `config/data.yaml`).
2. **Huấn luyện:** Fine-tune mô hình phân đoạn làn đường **`yolo11n-seg.pt`** (tự động chọn GPU CUDA hoặc CPU).
3. **Đầu ra mô hình & dự đoán (`data/output/`):**
   - Trọng số mô hình lưu tại **`data/output/yolo_train/weights/best.pt`** và **`data/output/best.pt`**.
   - Ảnh dự đoán kiểm thử (kèm nội suy B-Spline & phân loại làn đường UK Left-Hand Traffic) lưu tại **`data/output/predictions/`**.
4. **Xuất CSV số liệu & Biểu đồ tại thư mục gốc (`ROOT_DIR`):**
   - `training_results.csv`: Lịch sử loss và mAP qua các epoch.
   - `evaluation_metrics.csv`: Số liệu đánh giá trên tập validation (IoU, F1, góc lái UK, độ cong, thời gian suy luận).
   - `training_and_evaluation_chart.png`: Biểu đồ tổng hợp quá trình huấn luyện và đánh giá.

In [1]:
# ===== Cell 1: Khởi tạo môi trường & Kiểm tra đường dẫn =====
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

ROOT_DIR = Path.cwd().parent if (Path.cwd().parent / "README.md").exists() else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.path import RAW_DATA_DIR, PROCESSING_DATA_DIR, OUTPUT_DIR

train_device = 0 if torch.cuda.is_available() else "cpu"
print(f"Project Root         : {ROOT_DIR}")
print(f"Processing Input Dir : {PROCESSING_DATA_DIR}")
print(f"Output Directory     : {OUTPUT_DIR}")
print(f"Training Device      : {train_device}")

Project Root         : h:\FPT\4\CPV\project\CPV301
Processing Input Dir : h:\FPT\4\CPV\project\CPV301\data\processing
Output Directory     : h:\FPT\4\CPV\project\CPV301\data\output
Training Device      : cpu


## Bước 1 — Kiểm tra dữ liệu đầu vào trong `data/processing/`

Nếu `data/processing/train/images` chưa có dữ liệu, hệ thống sẽ tự động gọi `run_batch_preprocessing()` để tạo dữ liệu từ `data/raw/`.

In [2]:
# ===== Cell 2: Đảm bảo dữ liệu trong data/processing/ sẵn sàng =====
train_img_dir = PROCESSING_DATA_DIR / "train" / "images"
valid_img_dir = PROCESSING_DATA_DIR / "valid" / "images"

if not train_img_dir.exists() or not list(train_img_dir.glob("*.jpg")):
    print("Chưa thấy ảnh trong data/processing/train/images -> Đang chạy bước tiền xử lý tự động...")
    from src.preprocessing.batch_processor import run_batch_preprocessing
    run_batch_preprocessing(
        raw_dir=RAW_DATA_DIR,
        processing_dir=PROCESSING_DATA_DIR,
        root_dir=ROOT_DIR,
        max_highlight_images=60,
        max_yolo_clips=300,
        val_ratio=0.2,
    )

for split in ("train", "valid"):
    imgs = sorted((PROCESSING_DATA_DIR / split / "images").glob("*.jpg"))
    lbls = sorted((PROCESSING_DATA_DIR / split / "labels").glob("*.txt"))
    print(f"{split:5s} (data/processing/{split}): {len(imgs)} ảnh | {len(lbls)} nhãn YOLO-seg")

# Hiển thị 1 mẫu trong data/processing/train kèm nhãn đa giác làn đường
sample_imgs = sorted(train_img_dir.glob("*.jpg"))
assert sample_imgs, "Không tìm thấy ảnh trong data/processing/train/images!"
sample = sample_imgs[0]
img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
overlay = img.copy()
label_file = PROCESSING_DATA_DIR / "train" / "labels" / f"{sample.stem}.txt"
if label_file.exists():
    for line in label_file.read_text(encoding="utf-8").strip().splitlines():
        vals = list(map(float, line.split()))
        pts = vals[1:]
        poly = np.array([(int(pts[i] * w), int(pts[i + 1] * h)) for i in range(0, len(pts), 2)], dtype=np.int32)
        if len(poly) >= 3:
            cv2.fillPoly(overlay, [poly], (0, 255, 0))

plt.figure(figsize=(10, 5))
plt.imshow(cv2.addWeighted(img, 0.65, overlay, 0.35, 0))
plt.title(f"Mẫu huấn luyện từ data/processing: {sample.name}")
plt.axis("off")
plt.show()

train (data/processing/train): 256 ảnh | 256 nhãn YOLO-seg
valid (data/processing/valid): 60 ảnh | 60 nhãn YOLO-seg


C:\Users\pc\AppData\Local\Temp\ipykernel_6756\2095500150.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Bước 2 — Huấn luyện YOLO11-seg trên `data/processing/` và Xuất Kết quả ra `data/output/` + Root

Hàm `run_training_and_export()` thực hiện huấn luyện YOLO11-seg với dữ liệu trong `data/processing/`, lưu toàn bộ trọng số & ảnh dự đoán vào `data/output/`, đồng thời xuất `training_results.csv`, `evaluation_metrics.csv` và `training_and_evaluation_chart.png` ra thư mục gốc (`ROOT_DIR`).

In [3]:
# ===== Cell 3: Huấn luyện mô hình & Xuất kết quả ra data/output + Root =====
from src.evaluation.train_and_export import run_training_and_export

# Có thể tăng epochs=30..50 khi chạy trên máy có GPU CUDA
EPOCHS = 15 if torch.cuda.is_available() else 2
IMGSZ = 640 if torch.cuda.is_available() else 320

train_summary = run_training_and_export(
    processing_dir=PROCESSING_DATA_DIR,
    output_dir=OUTPUT_DIR,
    root_dir=ROOT_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=8,
    max_eval_images=30,
)

print("=== TỔNG HỢP KẾT QUẢ HUẤN LUYỆN & ĐÁNH GIÁ ===")
for k, v in train_summary.items():
    print(f"  {k:28s}: {v}")

New https://pypi.org/project/ultralytics/8.4.184 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.182  Python-3.11.0 torch-2.1.2+cpu CPU (Intel Core i5-10300H 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=h:\FPT\4\CPV\project\CPV301\config\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=h:\F

In [4]:
# ===== Cell 4: Kiểm tra bảng số liệu CSV và ảnh dự đoán trong data/output/ =====
eval_csv = ROOT_DIR / "evaluation_metrics.csv"
df_eval = pd.read_csv(eval_csv)
print(f"Bảng đánh giá tại Root: {eval_csv} ({len(df_eval)} mẫu kiểm thử)")
display(df_eval.head(10))

# Hiển thị 3 ảnh dự đoán từ data/output/predictions/
pred_imgs = sorted((OUTPUT_DIR / "predictions").glob("pred_*.jpg"))[:3]
if pred_imgs:
    fig, axes = plt.subplots(1, len(pred_imgs), figsize=(18, 5))
    if len(pred_imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, pred_imgs):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.name, fontsize=10)
        ax.axis("off")
    plt.suptitle("Kết quả dự đoán + Phân loại làn đường UK trong data/output/predictions/", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

Bảng đánh giá tại Root: h:\FPT\4\CPV\project\CPV301\evaluation_metrics.csv (3 mẫu kiểm thử)


,method,dataset_source,uk_traffic_mode,accuracy,fp,fn,fps
0,Hough_Transform,data/processing,left_hand_traffic,0.5000,0.0000,0.3333,6.56
1,LaneNet,data/processing,left_hand_traffic,0.7889,0.3333,0.3333,7.70
2,Proposed_Spline,data/processing,left_hand_traffic,1.0000,0.0000,0.0000,8.32


C:\Users\pc\AppData\Local\Temp\ipykernel_6756\532618171.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Bước 3 — Tích hợp Bản đồ `fh4_map` + Định vị `fh4_telemetry` + Tự hành làn trái Vương Quốc Anh (UK)

Sau khi huấn luyện và kiểm thử mô hình trong `data/output/`, bạn có thể chạy hệ thống tự lái hoàn chỉnh kết hợp:
- Phân tích biểu đồ đường đi trên bản đồ `fh4_map/raw/` -> `fh4_map/processing/`.
- Đánh dấu điểm đến (`--pick-dest` click chuột trên bản đồ hoặc `--dest-node 12`).
- Nhận tọa độ thực tế (`PositionX`, `PositionY`, `PositionZ`, `Yaw`) từ `fh4_telemetry.py`.
- Điều khiển xe chạy đúng **làn bên trái chuẩn Vương Quốc Anh (UK Left-Hand Traffic)** tới đích.

In [5]:
# ===== Cell 5: In lệnh khởi chạy hệ thống tự lái FH4 kết hợp Bản đồ & Làn đường UK =====
best_weights = OUTPUT_DIR / "best.pt"
print("1. Chạy tự lái kèm chọn điểm đến trực tiếp trên bản đồ FH4 (Click chuột):")
print(f"   python main.py --detector yolo --weights {best_weights} --pick-dest")
print("\n2. Chạy tự lái tới nút giao đích số 12 trên bản đồ UK (chế độ Hough siêu tốc):")
print("   python main.py --detector hough --dest-node 12")
print("\n3. Chạy mô phỏng kiểm thử trên video (View-Only) và xuất video ra data/output/:")
print("   python main.py --view-only --detector hough --dest-node 12 --video data/raw/test_driving.mp4 --save data/output/fh4_uk_autonomous.mp4")

1. Chạy tự lái kèm chọn điểm đến trực tiếp trên bản đồ FH4 (Click chuột):
   python main.py --detector yolo --weights h:\FPT\4\CPV\project\CPV301\data\output\best.pt --pick-dest

2. Chạy tự lái tới nút giao đích số 12 trên bản đồ UK (chế độ Hough siêu tốc):
   python main.py --detector hough --dest-node 12

3. Chạy mô phỏng kiểm thử trên video (View-Only) và xuất video ra data/output/:
   python main.py --view-only --detector hough --dest-node 12 --video data/raw/test_driving.mp4 --save data/output/fh4_uk_autonomous.mp4
